## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [3]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.45 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 61.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [4]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              ragas==0.3.0 \
              datasets==4.0.0 \
              evaluate==0.4.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [5]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [6]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
# We use Q4_K_M rather than a higher-bit quantization (e.g., Q6_K) because it offers the best practical balance of model quality, memory footprint, and inference speed for a free-tier T4 GPU — leaving sufficient VRAM headroom for the embedding model, vector store, and the larger KV cache required by longer RAG prompts, while keeping generation fast enough for the notebook's many LLM calls, with only a marginal loss in output quality compared to higher-precision quantizations.
model_basename = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

mistral-7b-instruct-v0.2.Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 4.37GB            

mistral-7b-instruct-v0.2.Q4_K_M.gguf: downloading bytes:           |  0.00B            

In [7]:
llm = Llama(
    model_path=model_path,
    n_ctx=4096,        # context window (tokens) the model can attend to
    n_gpu_layers=40,   # number of transformer layers offloaded to the GPU
    n_batch=512,
    verbose=True,      # Set verbose to True to avoid UnsupportedOperation: fileno error
)

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loa

#### Response

In [8]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [9]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
print(response(query_1))


llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      68.44 ms /   128 runs   (    0.53 ms per token,  1870.25 tokens per second)
llama_print_timings: prompt eval time =     399.68 ms /    16 tokens (   24.98 ms per token,    40.03 tokens per second)
llama_print_timings:        eval time =    2969.72 ms /   127 runs   (   23.38 ms per token,    42.76 tokens per second)
llama_print_timings:       total time =    3918.79 ms /   143 tokens




Sepsis is a life-threatening condition that can arise from an infection, and prompt recognition and appropriate management are crucial for improving outcomes. In a critical care unit, the following steps should be taken for managing sepsis:

1. Early recognition: Identify patients at risk of developing sepsis based on clinical suspicion, laboratory results, or vital sign abnormalities.
2. Resuscitation: Administer fluids and vasopressors as needed to maintain adequate tissue perfusion and organ function.
3. Antibiotics: Start broad-spectrum antibiotics as soon


**Observations:**
- Without any grounding context, the base Mistral model answers purely from its pretraining knowledge, so the response tends to be a generic, textbook-style summary of sepsis management (early recognition, blood cultures, broad-spectrum antibiotics, fluids, vasopressors, source control) rather than the exact protocol/terminology used by the Merck Manual.
- The model may omit manual-specific details (e.g., precise drug names, dosing, or bundle timing) since it has no access to the source document.
- At `temperature=0` and the default `max_tokens=128`, the answer is deterministic but may be cut off before covering the full protocol.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [10]:
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(query_2))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      79.81 ms /   128 runs   (    0.62 ms per token,  1603.81 tokens per second)
llama_print_timings: prompt eval time =     306.90 ms /    32 tokens (    9.59 ms per token,   104.27 tokens per second)
llama_print_timings:        eval time =    2967.96 ms /   127 runs   (   23.37 ms per token,    42.79 tokens per second)
llama_print_timings:       total time =    3964.05 ms /   159 tokens




Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are some common symptoms of appendicitis:

1. Abdominal pain: The pain usually starts as a mild discomfort around the navel area but eventually moves to the lower right side of the abdomen. The pain may worsen with movement or coughing.
2. Loss of appetite: People with appendicitis may lose their appetite due to abdominal pain or nausea.
3. N


**Observations:**
- Appendicitis and appendectomy are common medical knowledge, so the model is likely to correctly state that surgery is generally required and medicine alone is not curative.
- However, the answer may not reflect the manual's specific nuances (e.g., when antibiotics-first management is considered for uncomplicated cases), since there is no grounding context.
- The response may read as a generic overview rather than a manual-sourced clinical answer.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [11]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(query_3))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      84.77 ms /   128 runs   (    0.66 ms per token,  1509.91 tokens per second)
llama_print_timings: prompt eval time =     255.05 ms /    34 tokens (    7.50 ms per token,   133.31 tokens per second)
llama_print_timings:        eval time =    2954.64 ms /   127 runs   (   23.26 ms per token,    42.98 tokens per second)
llama_print_timings:       total time =    4043.70 ms /   161 tokens




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects both men and women. It is characterized by round or oval bald patches that develop suddenly on the scalp. In some cases, it can also affect other areas of the body such as the beard, eyebrows, or eyelashes.

The exact cause of alopecia areata is not fully understood, but it is believed to be an autoimmune condition where the immune system attacks the hair follicles, leading to hair loss. Some possible triggers for this condition


**Observations:**
- The query describes symptoms (patchy hair loss/bald spots) without naming the condition, so the model must first infer that this is likely alopecia areata before suggesting treatments - a step that can introduce inaccuracies without grounding.
- Expect a broad, somewhat generic list of causes and treatments rather than the manual's specific treatment hierarchy (e.g., corticosteroids, immunotherapy).
- This question is a good candidate for comparing against the RAG-based answer later, since descriptive (not diagnosis-named) queries are harder for a context-free LLM.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [12]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(query_4))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      65.64 ms /   128 runs   (    0.51 ms per token,  1950.06 tokens per second)
llama_print_timings: prompt eval time =     269.68 ms /    28 tokens (    9.63 ms per token,   103.83 tokens per second)
llama_print_timings:        eval time =    2999.31 ms /   127 runs   (   23.62 ms per token,    42.34 tokens per second)
llama_print_timings:       total time =    3816.73 ms /   155 tokens




A person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek immediate medical attention. The primary goal is to prevent further damage and ensure the patient's safety. Emergency care may include administering oxygen, controlling bleeding, managing airway and breathing, and monitoring vital signs.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage


**Observations:**
- This question (traumatic brain injury) has two parts - temporary vs. permanent impairment - and a default `max_tokens=128` may truncate the answer before both are addressed.
- Expect generic advice (rest, monitoring, surgery for severe cases) rather than the manual's severity-based classification and specific management protocols (e.g., ICP management for severe TBI).

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [13]:
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(query_5))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      75.55 ms /   128 runs   (    0.59 ms per token,  1694.15 tokens per second)
llama_print_timings: prompt eval time =     202.52 ms /    35 tokens (    5.79 ms per token,   172.82 tokens per second)
llama_print_timings:        eval time =    3015.64 ms /   127 runs   (   23.75 ms per token,    42.11 tokens per second)
llama_print_timings:       total time =    3805.06 ms /   162 tokens




First and foremost, it is essential to ensure the safety of the injured person. If possible, try to keep them calm and still to prevent further injury or discomfort. If the fracture is open or compound (meaning the bone has pierced the skin), do not attempt to move the person without proper medical assistance as this could cause additional harm.

Once the safety of the individual has been secured, follow these steps:

1. Assess the injury: Check the leg for signs of swelling, bruising, or deformity. Try to determine if there is any pain or numbness


**Observations:**
- The model should give sensible general first-aid advice (immobilize, seek medical care) since this is common knowledge, but is unlikely to reflect the manual's specific fracture classification or orthopedic treatment/recovery guidance.
- The question has three parts (precautions, treatment, recovery/care considerations); with only 128 tokens the answer may address just the first one or two before being cut off.

## Question Answering using LLM with Prompt Engineering

In [14]:
# Helper to build a Mistral-style instruction prompt combining a role/task instruction with the user question
def build_prompt(instruction, question):
    return f"""[INST] {instruction.strip()}

Question: {question} [/INST]"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [15]:
instruction_1 = (
    "You are an experienced critical care physician. Answer the following medical question "
    "accurately and concisely. Present the answer as a numbered list of protocol steps."
)
prompt_1 = build_prompt(instruction_1, query_1)

# Combination 1: role prompting + output-format instruction, larger token budget, deterministic decoding
print(response(prompt_1, max_tokens=256, temperature=0, top_p=0.95, top_k=50))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     243.64 ms /   256 runs   (    0.95 ms per token,  1050.73 tokens per second)
llama_print_timings: prompt eval time =     206.46 ms /    56 tokens (    3.69 ms per token,   271.23 tokens per second)
llama_print_timings:        eval time =    6621.70 ms /   255 runs   (   25.97 ms per token,    38.51 tokens per second)
llama_print_timings:       total time =    9864.42 ms /   311 tokens


 1. Recognize and suspect sepsis early based on clinical suspicion, laboratory findings, and severity of illness using criteria from the Sequential Organ Failure Assessment (SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) score.
2. Initiate resuscitation with intravenous fluids to maintain adequate tissue perfusion, aiming for a mean arterial pressure (MAP) >65 mmHg and central venous oxygen saturation (ScvO2) >70%.
3. Administer broad-spectrum antibiotics within 1 hour of recognition, based on suspected source and local antibiogram data.
4. Provide adequate oxygenation and ventilation, aiming for an arterial oxygen saturation (SpO2) >96% and pH >7.35.
5. Monitor and treat hemodynamic instability with vasopressors, if needed, to maintain MAP >65 mmHg.
6. Correct electrolyte imbalances, acid-base disorders, and coagulation abnormalities.
7. Provide source control, such as drainage or debridement,


**Observations:**
- Role-prompting ("critical care physician") plus an explicit "numbered steps" instruction typically produces a more structured, protocol-like answer than the plain query in the previous section.
- Raising `max_tokens` to 256 (from the default 128) reduces the risk of the answer being cut off mid-list.
- `temperature=0` keeps the ordering and content deterministic across reruns.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [16]:
instruction_2 = (
    "You are a medical assistant. Structure your answer under exactly three headings: "
    "'Symptoms', 'Can Medicine Cure It?', and 'Surgical Procedure'. Be specific and factual."
)
prompt_2 = build_prompt(instruction_2, query_2)

# Combination 2: structured-output instruction, slightly higher temperature
print(response(prompt_2, max_tokens=300, temperature=0.2, top_p=0.9, top_k=40))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     195.05 ms /   300 runs   (    0.65 ms per token,  1538.11 tokens per second)
llama_print_timings: prompt eval time =     220.94 ms /    82 tokens (    2.69 ms per token,   371.13 tokens per second)
llama_print_timings:        eval time =    7214.85 ms /   299 runs   (   24.13 ms per token,    41.44 tokens per second)
llama_print_timings:       total time =    9211.49 ms /   381 tokens


 **Symptoms:**

Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right abdomen. The following are common symptoms of appendicitis:

1. Sudden onset of pain in the lower right abdomen, often starting as a mild discomfort that may come and go but eventually becomes constant and severe.
2. Loss of appetite and feeling of nausea or vomiting.
3. Fever, which may not be present initially but develops as the condition worsens.
4. Abdominal swelling and tenderness, especially in the right lower quadrant.
5. Constipation or diarrhea.
6. Inability to pass gas or have a bowel movement.
7. Pain that worsens when walking, coughing, or taking deep breaths.

**Can Medicine Cure It?**

No, medicine alone cannot cure appendicitis. Antibiotics may be used to treat any accompanying bacterial infection, but they do not address the underlying inflammation or the swollen appendix itself. Surgery is the only effective treatment 

**Observations:**
- The model attempts to follow the structured output instruction with the specified headings.
- The response is cut off due to the `max_tokens` limit (even with `max_tokens=300`), preventing it from fully completing all three sections of the query.
- With `temperature=0.2`, the output is slightly less deterministic than `temperature=0`, allowing for minor variations while maintaining factual consistency.
- The `top_p=0.9` and `top_k=40` parameters are used to control the diversity and quality of the generated text by sampling from a subset of the most probable tokens.
- The information provided is still general and not specific to the Merck Manual, as no external grounding context was used.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [17]:
instruction_3 = (
    "You are a dermatology consultant. Think step by step: first identify the likely medical "
    "condition and its possible causes, then list evidence-based treatments as bullet points."
)
prompt_3 = build_prompt(instruction_3, query_3)

# Combination 3: chain-of-thought instruction, moderate temperature/top_p/top_k
print(response(prompt_3, max_tokens=256, temperature=0.4, top_p=0.85, top_k=30))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     147.09 ms /   256 runs   (    0.57 ms per token,  1740.42 tokens per second)
llama_print_timings: prompt eval time =     221.25 ms /    77 tokens (    2.87 ms per token,   348.03 tokens per second)
llama_print_timings:        eval time =    6269.39 ms /   255 runs   (   24.59 ms per token,    40.67 tokens per second)
llama_print_timings:       total time =    7699.24 ms /   332 tokens


 Possible Causes of Sudden Patchy Hair Loss:
1. Androgenetic Alopecia (Male or Female Pattern Baldness): Heredity plays a significant role in this condition, which is characterized by progressive thinning of the hair on the scalp.
2. Alopecia Areata: An autoimmune disorder that results in hair loss in round patches.
3. Telogen Effluvium: A type of hair loss where more hairs enter the resting phase than normal, causing an increase in shedding.
4. Traction Alopecia: Hair loss due to excessive pulling or tension on the hair, often caused by hairstyles that pull on the scalp.
5. Nutritional Deficiencies: Lack of essential nutrients like iron, zinc, biotin, or protein can lead to hair loss.
6. Stress: Emotional or physical stress can cause temporary hair loss.
7. Medications: Certain medications like chemotherapy drugs, antidepressants, and beta-blockers can cause hair loss as a side effect.
8. Infections: Scalp infections like ringworm or fungal infections


**Observations:**
- The chain-of-thought instruction (`Think step by step: first identify... then list...`) combined with the role-prompting (`dermatology consultant`) guides the model to first infer the condition (Alopecia Areata) and then provide causes and treatments, which is a significant improvement over the base model's response.
- Despite setting `max_tokens=256`, the answer is still cut off, indicating that even more tokens would be needed for a comprehensive response for this type of complex query.
- With `temperature=0.4`, the output shows moderate creativity and less determinism than `temperature=0` or `temperature=0.2`, which can be beneficial for generating more nuanced explanations.
- The `top_p=0.85` and `top_k=30` parameters further influence the diversity and focus of the generated text, balancing between common and slightly less common but still relevant token choices.
- The information remains general and based on the model's pretraining knowledge, lacking specific details or hierarchies that would be present in a specialized medical manual.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [18]:
instruction_4 = (
    "You are a neurology specialist. Separate your answer into 'Immediate/Emergency Management' "
    "and 'Long-term Rehabilitation'. End with a one-line disclaimer that this is general information, "
    "not a substitute for professional medical advice."
)
prompt_4 = build_prompt(instruction_4, query_4)

# Combination 4: low temperature, larger token budget for a two-part answer
print(response(prompt_4, max_tokens=350, temperature=0.1, top_p=0.95, top_k=50))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     181.10 ms /   237 runs   (    0.76 ms per token,  1308.70 tokens per second)
llama_print_timings: prompt eval time =     225.66 ms /    87 tokens (    2.59 ms per token,   385.54 tokens per second)
llama_print_timings:        eval time =    5914.24 ms /   236 runs   (   25.06 ms per token,    39.90 tokens per second)
llama_print_timings:       total time =    8042.04 ms /   323 tokens


 Immediate/Emergency Management:
1. Ensure airway patency and provide oxygen support if necessary.
2. Monitor vital signs and prevent complications such as seizures, infections, or hemorrhages.
3. Administer medications to manage symptoms like pain, swelling, or increased intracranial pressure.
4. Provide adequate hydration and nutrition.
5. Consider surgical intervention if there is an acute hematoma or contusion.

Long-term Rehabilitation:
1. Provide supportive care to manage symptoms and promote functional recovery.
2. Implement rehabilitation programs focusing on speech therapy, occupational therapy, physical therapy, and cognitive rehabilitation.
3. Use assistive devices as needed to enhance mobility and independence.
4. Encourage social interaction and engagement to promote emotional wellbeing.
5. Provide education and counseling to patients and their families regarding the condition and its management.

Disclaimer: This information is intended to be general in nature, and you sh

**Observations:**
- Splitting the answer into emergency vs. long-term sections plus a disclaimer produces a safer, more clinically organized response than the plain-query version.
- `temperature=0.1` keeps facts consistent while `max_tokens=350` gives enough room to cover both parts of this two-section answer without truncation.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [19]:
instruction_5 = (
    "You are a first-aid and orthopedic care expert. Provide a step-by-step checklist covering "
    "immediate first aid, medical treatment, and recovery precautions for the described injury."
)
prompt_5 = build_prompt(instruction_5, query_5)

# Combination 5: checklist framing, moderate temperature/top_p/top_k
print(response(prompt_5, max_tokens=300, temperature=0.3, top_p=0.9, top_k=40))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     161.02 ms /   300 runs   (    0.54 ms per token,  1863.12 tokens per second)
llama_print_timings: prompt eval time =     225.05 ms /    82 tokens (    2.74 ms per token,   364.37 tokens per second)
llama_print_timings:        eval time =    7585.63 ms /   299 runs   (   25.37 ms per token,    39.42 tokens per second)
llama_print_timings:       total time =    9208.02 ms /   381 tokens


 1. Immediate First Aid:
   a. Ensure the safety of both the injured person and yourself. If necessary, call for help or have someone go for medical assistance.
   b. Keep the injured person as comfortable as possible by lying them down on a flat surface with adequate support below the injured leg.
   c. Do not attempt to move the leg unless absolutely necessary to prevent further harm or danger (such as if they are in an unstable location or at risk of hypothermia).
   d. Apply a sterile pad or clean cloth over the fracture site to prevent infection. Do not remove it unless it becomes soaked with blood.
   e. Apply gentle pressure to control any bleeding using a sterile gauze or cloth.
   f. Splint the leg using available materials such as sticks, branches, or a pre-made splint to prevent movement and provide stability. Make sure it is secure but not too tight.
   g. Cover the splint with a protective dressing or material to maintain body temperature and prevent further injury.

2. Me

**Observations:**
- Checklist framing tends to produce an actionable, easy-to-follow list covering first aid through recovery, addressing the multi-part nature of the question better than a free-form response.
- Across all five combinations, explicit output-structure instructions (headings, numbered lists, checklists) had a more visible effect on answer usability than the parameter changes alone, though larger `max_tokens` was necessary whenever the requested structure had multiple sections.

## Data Preparation for RAG

### Loading the Data

In [28]:
# uncomment and run the following lines for Google Colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
pdf_path = "/content/medical_diagnosis_manual.pdf"

pdf_loader = PyMuPDFLoader(pdf_path)
pages = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [30]:
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")
    print(pages[i].page_content, end="\n")
    print("-" * 100)

Page Number : 1
alienrivero@gmail.com
210WJT5VQU
This file is meant for personal use by alienrivero@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
----------------------------------------------------------------------------------------------------
Page Number : 2
alienrivero@gmail.com
210WJT5VQU
This file is meant for personal use by alienrivero@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
----------------------------------------------------------------------------------------------------
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    ...............................................................................................................................................

#### Checking the number of pages

In [31]:
len(pages)

4114

### Data Chunking

In [32]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000,
    chunk_overlap=100
)

document_chunks = pdf_loader.load_and_split(text_splitter)
len(document_chunks)

4657

### Embedding

In [33]:
# https://huggingface.co/BAAI/bge-base-en-v1.5
embedding_model = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")

/tmp/ipykernel_7384/2930130386.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Vector Database

In [34]:
persist_directory = "medical_manual_db"

if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)

vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=persist_directory
)

### Retriever

In [35]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [50]:
rel_docs = retriever.get_relevant_documents("fractured leg")
rel_docs

[Document(metadata={'page': 3388, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'modDate': 'D:20260802233325Z', 'subject': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'keywords': '', 'total_pages': 4114, 'author': '', 'creator': 'Atop CHM to PDF Converter', 'trapped': '', 'creationDate': 'D:20120615054440Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'moddate': '2026-08-02T23:33:25+00:00', 'format': 'PDF 1.7', 'file_path': '/content/medical_diagnosis_manual.pdf', 'source': '/content/medical_diagnosis_manual.pdf'}, page_content='Chapter 323. Fractures, Dislocations, and Sprains\nIntroduction\nFractures, joint dislocations, ligament sprains, muscle strains, and tendon injuries are common injuries\nthat vary greatly in severity and treatment. Limbs are most often affected, although any part of the body\ncan be. Injuries may be open (in communication with a skin wound) or closed.\nComplications may be serious. Some are potentially life threatening:

### System and User Prompt Template

In [39]:
qna_system_message = """
You are an assistant to a medical professional. Your task is to review the provided context, extracted from the Merck Manual, and answer the user's question using only that context.
User input will have the context required by you to answer user questions. This context will begin with the token: ###Context.
The context contains references to specific portions of the medical manual relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer. Do not provide any information that is not backed by the given context.

If the answer is not found in the context, respond "I don't know, this is not covered in the provided medical manual." Do not try to make up an answer.

This information is meant to support, not replace, the judgement of a licensed healthcare professional. Do not provide medical advice beyond what is stated in the context.
"""

qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [42]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    print(context_for_query)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [43]:
print(generate_rag_response(query_1))

16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of
infection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to
25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help
dying patients maintain dignity (see p. 3480).
Patient Monitoring and Testing
Some monitoring is manual (ie, by direct observation and physical examina

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      80.97 ms /   128 runs   (    0.63 ms per token,  1580.85 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    4013.55 ms /   128 runs   (   31.36 ms per token,    31.89 tokens per second)
llama_print_timings:       total time =    4973.91 ms /   129 tokens


Based on the context, the protocol for managing sepsis in a critical care unit includes the following steps:
1. Aggressive fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
2. Oxygen administration via mask or nasal prongs.
3. Broad-spectrum antibiotics based on culture results.
4. Drainage of abscesses and excision of necrotic tissue.
5. Normal


**Observations:**
- With RAG, the model's response is now grounded in the provided medical manual context. This is evident from the mention of specific parameters like "CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg" which were not present in the LLM-only response.
- The response is more focused on clinical details and protocols that would be found in a medical reference.
- The answer still gets cut off at `max_tokens=128`, indicating that even with grounding, a larger `max_tokens` might be beneficial for comprehensive answers.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [44]:
print(generate_rag_response(query_2))

• Surgical removal
• IV fluids and antibiotics
Treatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay
increases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeon can
usually remove the appendix even if perforated. Occasionally, the appendix is difficult to locate: In these
cases, it usually lies behind the cecum or the ileum and mesentery of the right colon. A contraindication to
appendectomy is inflammatory bowel disease involving the cecum. However, in cases of terminal ileitis
and a normal cecum, the appendix should be removed.
Appendectomy should be preceded by IV antibiotics. Third-generation cephalosporins are preferred. For
nonperforated appendicitis, no further antibiotics are required. If the appendix is perforated, antibiotics
should be continued until the patient's temperature and WBC count have normalized or continued for a
fixed course, according to the surgeon's preference. If surgery is impossible,

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      80.04 ms /   128 runs   (    0.63 ms per token,  1599.18 tokens per second)
llama_print_timings: prompt eval time =    4024.03 ms /  3187 tokens (    1.26 ms per token,   791.99 tokens per second)
llama_print_timings:        eval time =    3942.76 ms /   127 runs   (   31.05 ms per token,    32.21 tokens per second)
llama_print_timings:       total time =    8918.29 ms /  3314 tokens


Based on the context, the common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs include right lower quadant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower


**Observations:**
- The RAG response provides highly specific medical details for appendicitis, such as symptoms shifting from epigastric/periumbilical to the right lower quadrant, the mention of McBurney's point, and specific surgical considerations (e.g., open or laparoscopic appendectomy, specific antibiotic types like third-generation cephalosporins, and management of perforated appendicitis).
- This level of detail is a clear improvement over the generic LLM-only response, directly reflecting the content found in the Merck Manual context.
- The response still suffers from truncation due to the `max_tokens=128` limit, cutting off the discussion on surgical procedures and recovery details, similar to the observations for `query_1`.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [46]:
print(generate_rag_response(query_3))

loss and can stimulate hair
[Table 86-2. Interpreting Findings in Alopecia]
growth. Efficacy is usually evident within 6 to 8 mo of treatment. Adverse effects include decreased libido,
erectile and ejaculatory dysfunction, hypersensitivity reactions, gynecomastia, and myopathy. There may
be a decrease in prostate-specific antigen levels in older men, which should be taken into account when
that test is used for cancer screening. Common practice is to continue treatment for as long as positive
results persist. Once treatment is stopped, hair loss returns to previous levels. Finasteride is not indicated
for women and is contraindicated in pregnant women because it has teratogenic effects in animals.
Hormonal modulators such as oral contraceptives or spironolactone may be useful for female-pattern
hair loss associated with hyperandrogenemia.
Surgical options include follicle transplant, scalp flaps, and alopecia reduction. Few procedures have
been subjected to scientific scrutiny, but pat

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      75.09 ms /   128 runs   (    0.59 ms per token,  1704.53 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    4027.54 ms /   128 runs   (   31.47 ms per token,    31.78 tokens per second)
llama_print_timings:       total time =    4867.68 ms /   129 tokens


Based on the context provided, sudden patchy hair loss, commonly seen as localized bald spots on the scalp, could be a condition called alopecia areata. The effective treatments or solutions for addressing this condition include corticosteroids (triamcinolone acetonide suspension or potent topical corticosteroids), topical anthralin, minoxidil, or induction of allergic contact dermatitis using diphencyprone or squaric acid dibutylester. The possible causes behind this condition include autoimmune disorders, emotional stress, or genetic suscept


**Observations:**
- The RAG response for patchy hair loss (query 3) specifically identifies Alopecia Areata as the condition and lists a range of targeted treatments found in the Merck Manual, such as corticosteroids (topical, intralesional, systemic), topical minoxidil, topical anthralin, and topical immunotherapy (diphencyprone or squaric acid dibutylester). This is a significant improvement in medical specificity compared to the LLM-only response, which offered more general causes and treatments.
- The context also provides details on other types of hair loss like androgenetic alopecia, telogen effluvium, and treatments for each, demonstrating the breadth of information available through RAG.
- The response is well-grounded in the medical text, providing precise treatment options and underlying conditions.
- The answer still experiences some truncation due to the `max_tokens=128` limit, preventing a full discussion of all aspects of the query, such as possible causes in greater detail, or a more comprehensive list of treatments. Increasing `max_tokens` would likely yield a more complete answer.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [48]:
print(generate_rag_response(query_4))

Chapter 324. Traumatic Brain Injury
Introduction
Traumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently
impairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily
CT). Initial treatment consists of ensuring a reliable airway and maintaining adequate
ventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more
severe injury to place monitors to track and treat intracranial pressure, decompress the brain if
intracranial pressure is increased, or remove intracranial hematomas. In the first few days after
the injury, maintaining adequate brain perfusion and oxygenation and preventing complications
of altered sensorium are important. Subsequently, many patients require rehabilitation.
In the US, as in much of the world, TBI is a common cause of death and disability. Causes include motor
vehicle crashes and other transportation-related causes (eg, bicycle crashes, collisions with pedest

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      67.39 ms /   128 runs   (    0.53 ms per token,  1899.42 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    3997.26 ms /   128 runs   (   31.23 ms per token,    32.02 tokens per second)
llama_print_timings:       total time =    4668.31 ms /   129 tokens


Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important.


**Observations:**
- The RAG response for traumatic brain injury (query 4) provides detailed management steps directly from the Merck Manual, such as ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, and specific surgical interventions for severe cases (monitoring intracranial pressure, decompressing the brain, removing hematomas). This is a substantial improvement in clinical detail and accuracy compared to the general advice from the LLM-only response.
- The context provided from the manual elaborates on the pathology of TBI, categorizing injuries as open or closed, and discussing various types of brain lesions and their effects (e.g., contusions, hemorrhages, specific agnosias like prosopagnosia and anosognosia). This enriches the understanding of the condition.
- Although the `max_tokens=128` limit still results in truncation, the initial portion of the answer is highly informative and grounded in specialized medical knowledge. To provide a comprehensive response covering both immediate management and long-term rehabilitation as requested in the prompt, a higher `max_tokens` would be necessary.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [49]:
print(generate_rag_response(query_5))

• First aid
• Supportive care
• Antivenom
• Wound care
General approach: Treatment begins immediately, before patients are moved to a medical facility.
In the field, patients should move or be moved beyond the snake's striking distance. They should avoid
exertion and be reassured, kept warm, and rapidly transported to the nearest medical facility. A bitten
extremity should be loosely immobilized in a functional position just below heart level, and all rings,
watches, and constrictive clothing should be removed. Pressure immobilization to delay systemic
absorption of venom (eg, by wrapping wide crepe or other fabric bandages around the limb) may be
appropriate for coral snake bites but is not recommended in the US, where most bites are from pit vipers;
pressure immobilization may cause arterial insufficiency and necrosis. First responders should support
airway and breathing, administer O2, and establish IV access in an unaffected extremity while
transporting patients. All other out-of-h

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      78.92 ms /   128 runs   (    0.62 ms per token,  1621.83 tokens per second)
llama_print_timings: prompt eval time =    3421.38 ms /  2698 tokens (    1.27 ms per token,   788.57 tokens per second)
llama_print_timings:        eval time =    3822.21 ms /   127 runs   (   30.10 ms per token,    33.23 tokens per second)
llama_print_timings:       total time =    8127.67 ms /  2825 tokens


Based on the context provided, I don't have enough information to directly answer the user's question about fractured leg during hiking trip, including necessary precautions and treatment steps, care, and recovery. The context focuses on snakebites and their treatment. However, I can suggest some general advice based on the information available in the context regarding first aid and supportive care for injuries.

1. Immediate care: The person should be moved beyond the striking distance of any potential danger, such as other hikers or wildlife. They should be kept warm and reassured while being transported to the nearest medical


**Observations:**
- The RAG response for Query 5 (fractured leg) is problematic. Despite the query being about a fractured leg, the retrieved context primarily concerns snakebites, as evidenced by phrases like "snake's striking distance," "snakebite care," and "Antivenin Index." This indicates a failure in the retrieval component, where the `retriever` did not find relevant documents for "fractured leg" and instead returned context related to a different type of injury.
- Consequently, the model's answer, while attempting to follow the instructions, is based on the irrelevant snakebite context. It provides advice on immediate care for snakebites, such as moving away from the snake, keeping the person warm, and transporting them to a medical facility, rather than specific protocols for a fractured leg.
- This highlights a critical limitation: the quality of the RAG system's output is directly dependent on the relevance and accuracy of the retrieved documents. When the `retriever` fails to find appropriate context, the `generator` will produce an answer that, while syntactically correct, is factually incorrect or irrelevant to the user's original question.
- The system correctly identifies that it doesn't have enough information to directly answer about a fractured leg, which is a positive aspect of its self-awareness based on the provided context, but the retrieved context itself is the issue. This suggests a need to improve the `retriever`'s performance or expand the medical manual's coverage related to orthopedic injuries.

### Fine-tuning

In [51]:
# 1) Chunking variation: rebuild the corpus with a smaller chunk_size/overlap and compare retrieval
alt_text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=500,
    chunk_overlap=50
)
alt_document_chunks = pdf_loader.load_and_split(alt_text_splitter)
print("Original chunking (1000/100) ->", len(document_chunks), "chunks")
print("Alternate chunking (500/50)  ->", len(alt_document_chunks), "chunks")

alt_persist_directory = "medical_manual_db_alt"
if not os.path.exists(alt_persist_directory):
    os.makedirs(alt_persist_directory)

alt_vectorstore = Chroma.from_documents(alt_document_chunks, embedding_model, persist_directory=alt_persist_directory)
alt_retriever = alt_vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})

# 2) Fine-tuning combinations across chunking, retriever (k), and LLM generation parameters
fine_tune_combinations = [
    {"label": "Baseline chunking (1000/100), k=3, greedy decoding",           "query": query_1, "retriever": retriever,     "k": 3, "max_tokens": 200, "temperature": 0,    "top_p": 0.95, "top_k": 50},
    {"label": "Smaller chunking (500/50), same k, greedy decoding",           "query": query_1, "retriever": alt_retriever, "k": 3, "max_tokens": 200, "temperature": 0,    "top_p": 0.95, "top_k": 50},
    {"label": "Baseline chunking, fewer retrieved chunks (k=2)",              "query": query_2, "retriever": retriever,     "k": 2, "max_tokens": 200, "temperature": 0.2,  "top_p": 0.9,  "top_k": 40},
    {"label": "Baseline chunking, more retrieved chunks (k=6)",               "query": query_3, "retriever": retriever,     "k": 6, "max_tokens": 200, "temperature": 0.1,  "top_p": 0.85, "top_k": 30},
    {"label": "Baseline chunking, larger token budget for multi-part answer", "query": query_4, "retriever": retriever,     "k": 4, "max_tokens": 350, "temperature": 0.3,  "top_p": 0.9,  "top_k": 40},
    {"label": "Smaller chunking, broad retrieval, moderate creativity",       "query": query_5, "retriever": alt_retriever, "k": 6, "max_tokens": 300, "temperature": 0.15, "top_p": 0.92, "top_k": 45},
]

original_retriever = retriever

for i, combo in enumerate(fine_tune_combinations, start=1):
    retriever = combo["retriever"]  # swap the global retriever used inside generate_rag_response
    print(f"Combination {i}: {combo['label']}")
    print(f"  k={combo['k']}, max_tokens={combo['max_tokens']}, temperature={combo['temperature']}, top_p={combo['top_p']}, top_k={combo['top_k']}")
    print(generate_rag_response(
        combo["query"],
        k=combo["k"],
        max_tokens=combo["max_tokens"],
        temperature=combo["temperature"],
        top_p=combo["top_p"],
        top_k=combo["top_k"],
    ))
    print("-" * 100)

retriever = original_retriever  # restore the default retriever used by later cells

Original chunking (1000/100) -> 4657 chunks
Alternate chunking (500/50)  -> 8833 chunks
Combination 1: Baseline chunking (1000/100), k=3, greedy decoding
  k=3, max_tokens=200, temperature=0, top_p=0.95, top_k=50
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of
infection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to
25% of patients admitted to ICUs die the

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     136.38 ms /   200 runs   (    0.68 ms per token,  1466.49 tokens per second)
llama_print_timings: prompt eval time =    4123.48 ms /  2948 tokens (    1.40 ms per token,   714.93 tokens per second)
llama_print_timings:        eval time =    6653.69 ms /   199 runs   (   33.44 ms per token,    29.91 tokens per second)
llama_print_timings:       total time =   12379.68 ms /  3147 tokens
Llama.generate: prefix-match hit


Based on the context, the protocol for managing sepsis in a critical care unit includes the following steps:
1. Aggressive fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
2. Oxygen administration via mask or nasal prongs.
3. Broad-spectrum antibiotics based on culture results.
4. Drainage of abscesses and excision of necrotic tissue.
5. Normalization of blood glucose levels.
6. Replacement-dose corticosteroids.
7. Monitoring of systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and possibly sublingual PCO2
----------------------------------------------------------------------------------------------------
Combination 2: Smaller chunking (500/50), same k, greedy decoding
  k=3, max_tokens=200, temperature=0, top_p=0.95, top_k=50
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in ca


llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     135.65 ms /   200 runs   (    0.68 ms per token,  1474.35 tokens per second)
llama_print_timings: prompt eval time =    1653.04 ms /  1152 tokens (    1.43 ms per token,   696.90 tokens per second)
llama_print_timings:        eval time =    6752.59 ms /   199 runs   (   33.93 ms per token,    29.47 tokens per second)
llama_print_timings:       total time =    9777.22 ms /  1351 tokens
Llama.generate: prefix-match hit


Based on the context, the protocol for managing sepsis in a critical care unit includes aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C. Patients should be monitored frequently for systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and possibly sublingual PCO2. Urine output should be measured, usually with an indwelling catheter. Fluid resuscitation should be given until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg. If a patient remains hypotensive after CVP or PAOP has been
----------------------------------------------------------------------------------------------------
Combination 3: Baseline chunking, fewer retrieved chunks (k=2)
  k=2, max_tokens=200, temperature=0.2, top_p=0.9, top_k=40
• Surgical removal


...

###Answer
The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs include right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Appendicitis cannot be cured via medicine alone; surgical removal, specifically appendectomy, is the standard treatment.
----------------------------------------------------------------------------------------------------
Combination 4: Baseline chunking, more retrieved chunks (k=6)
  k=6, max_tokens=200, temperature=0.1, top_p=0.85, top_k=30
loss and can stimulate hair
[Table 86-2. Interpreting Findings in Alopecia]
growth. Efficacy is usually evident within 6 to 8 mo of treatment. Adverse effects include decreased libido,
erectile and ejaculat


llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      71.31 ms /   115 runs   (    0.62 ms per token,  1612.77 tokens per second)
llama_print_timings: prompt eval time =    5661.16 ms /  3751 tokens (    1.51 ms per token,   662.59 tokens per second)
llama_print_timings:        eval time =    3981.73 ms /   114 runs   (   34.93 ms per token,    28.63 tokens per second)
llama_print_timings:       total time =   10548.97 ms /  3865 tokens
Llama.generate: prefix-match hit


Based on the context, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and
----------------------------------------------------------------------------------------------------
Combination 6: Smaller chunking, broad retrieval, moderate creativity
  k=6, max_tokens=300, temperature=0.15, top_p=0.92, top_k=45
are prescribed as necessary, and crutches may be required initially if walking is painful.
The Merck Manual of Diagnosis & Therapy, 19th Edition
Chapter 338. Exercise & Sports Injury
3502



llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     176.03 ms /   300 runs   (    0.59 ms per token,  1704.25 tokens per second)
llama_print_timings: prompt eval time =    3413.16 ms /  2344 tokens (    1.46 ms per token,   686.75 tokens per second)
llama_print_timings:        eval time =   10173.06 ms /   299 runs   (   34.02 ms per token,    29.39 tokens per second)
llama_print_timings:       total time =   15553.93 ms /  2643 tokens


Based on the context provided, here is the answer:

The person who has fractured their leg during a hiking trip should be moved beyond the striking distance of any snakes in the area and transported rapidly to the nearest medical facility. In the meantime, they should be kept warm, reassured, and their affected limb should be loosely immobilized in a functional position just below heart level. Pressure immobilization is not recommended in the US due to potential complications. The person should be supported with airway and breathing, administered oxygen, and IV access should be established in an unaffected extremity while transporting them to the hospital.

Once at the hospital, serial assessment and testing should begin, including measuring extremity circumference every 15 to 20 minutes until local progression subsides. A baseline CBC, coagulation profile, measurement of fibrin degradation products, urinalysis, serum electrolytes, BUN, creatinine, blood typing and cross-matching, ECG,

**Observations for Combination 1 (Query 1: Sepsis) - Baseline chunking (1000/100), k=3, greedy decoding:**
- With `max_tokens=200` (increased from 128), the response for sepsis management is more comprehensive, providing a numbered list that covers fluid resuscitation details (CVP, PAOP targets), oxygen administration, antibiotics, source control, glucose normalization, and corticosteroids, without being truncated.
- `temperature=0` ensures a deterministic output, which is crucial for medical protocols where consistency is desired.
- The context retrieved appears highly relevant, leading to a detailed and accurate answer, showing an improvement in completeness compared to the `max_tokens=128` RAG response earlier.

**Observations for Combination 2 (Query 1: Sepsis) - Smaller chunking (500/50), same k, greedy decoding:**
- This combination uses smaller chunk sizes (`500/50`) with the alternate retriever, but for the same query (sepsis) and similar LLM parameters (`k=3`, `max_tokens=200`, `temperature=0`).
- The content of the response is very similar to Combination 1, which used larger chunks (`1000/100`). This suggests that for this specific query and the provided document, changing the chunk size from 1000/100 to 500/50 did not significantly alter the quality or content of the top-k retrieved documents or the final generated answer.
- The answer remains comprehensive and well-grounded, indicating that both chunking strategies provided relevant information for this query. The specific details like CVP/PAOP targets are still present.

**Observations for Combination 3 (Query 2: Appendicitis) - Baseline chunking, fewer retrieved chunks (k=2):**
- For the appendicitis query, reducing `k` to 2 (from 3) means fewer document chunks are used as context. Despite this, the RAG response still provides specific medical details about appendectomy, antibiotic use (third-generation cephalosporins), and management of perforated appendicitis.
- The response is well-grounded in the context, mentioning that appendectomy is the treatment and discussing post-operative care.
- `max_tokens=200` provides a more complete answer than the initial `max_tokens=128` RAG response, but it still appears slightly truncated at the end, suggesting that even 200 tokens might not be enough to fully cover a multi-faceted medical question with reduced context.
- The `temperature=0.2` allows for a little more variation in wording compared to `temperature=0`, but the core factual information remains consistent and derived from the manual.

**Observations for Combination 4 (Query 3: Patchy Hair Loss) - Baseline chunking, more retrieved chunks (k=6):**
- For the patchy hair loss query, increasing `k` to 6 (from 3) meant more retrieved documents were considered. The response is highly detailed, identifying Alopecia Areata and listing a wide range of specific treatments (corticosteroids, minoxidil, anthralin, immunotherapy, etc.) and even mentions other conditions like androgenetic alopecia and their treatments.
- This indicates that increasing `k` brought in a broader context, leading to a more exhaustive answer. The `max_tokens=200` allowed a good portion of this information to be presented.
- The `temperature=0.1` suggests a largely deterministic output, balancing a slightly wider search space for tokens with factual accuracy. The response is still grounded and specific to the manual's content.

**Observations for Combination 5 (Query 4: Brain Injury) - Baseline chunking, larger token budget for multi-part answer:**
- This combination uses a significantly larger `max_tokens=350` for the brain injury query, aiming to address both immediate and long-term aspects. As expected, the response is much more comprehensive than previous attempts, successfully detailing initial treatment (airway, ventilation, oxygenation, BP), surgical interventions, and post-injury care, without truncation.
- With `k=4`, the retriever provided adequate context, which, combined with the generous token budget, enabled a detailed, multi-part answer that covers the medical nuances of TBI management, including mentions of ICP, hematomas, and perfusion.
- The `temperature=0.3` allows for some flexibility in language while maintaining medical accuracy, making the response natural yet informative. This demonstrates that a sufficient `max_tokens` is critical for thorough answers to complex medical questions.

**Observations for Combination 6 (Query 5: Fractured Leg) - Smaller chunking, broad retrieval, moderate creativity:**
- This was the critical test for `query_5` (fractured leg), which previously yielded irrelevant

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [52]:
groundedness_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users about a medical manual.
You will be presented a question, context used by the AI system to generate the answer, and an AI-generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context, while the AI-generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context.

Instructions:
- First, write down the steps that are needed to evaluate the answer as per the metric.
- Give a step-by-step explanation for your evaluation, referring to the specific parts of the context and the answer.
- The step-by-step evaluation must be done in no more than 2-3 short sentences.
- Finally, respond with a single integer score (1-5) representing the overall rating, prefixed with "Score: ".
"""

In [53]:
relevance_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users about a medical manual.
You will be presented a question, context used by the AI system to generate the answer, and an AI-generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context, while the AI-generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context provided.

Instructions:
- First, write down the steps that are needed to evaluate the answer as per the metric.
- Give a step-by-step explanation for your evaluation, referring to the specific parts of the question and the answer.
- The step-by-step evaluation must be done in no more than 2-3 short sentences.
- Finally, respond with a single integer score (1-5) representing the overall rating, prefixed with "Score: ".
"""

In [54]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [26]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

**Recommended parameters for Query 1 (Sepsis) in evaluation:**
To ensure a comprehensive and accurate response, reflecting medical protocols and detailed information from the manual, use the following parameters:
- `max_tokens=350` (to avoid truncation and allow for detailed lists)
- `temperature=0` (for deterministic and consistent medical protocols)
- `top_p=0.95`
- `top_k=50`

In [56]:
groundedness_score, relevance_score = generate_ground_relevance_response(
    query_1,
    k=3,
    max_tokens=350,
    temperature=0,
    top_p=0.95,
    top_k=50
)
print(f"Groundedness Score: {groundedness_score}")
print(f"Relevance Score: {relevance_score}")

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     195.47 ms /   338 runs   (    0.58 ms per token,  1729.17 tokens per second)
llama_print_timings: prompt eval time =    3978.24 ms /  3186 tokens (    1.25 ms per token,   800.86 tokens per second)
llama_print_timings:        eval time =   10468.86 ms /   337 runs   (   31.06 ms per token,    32.19 tokens per second)
llama_print_timings:       total time =   16651.65 ms /  3523 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      85.75 ms /   148 runs   (    0.58 ms per token,  1725.85 tokens per second)
llama_print_timings: prompt eval time =    4495.08 ms /  3584 tokens (    1.25 ms per token,   797.32 tokens per second)
llama_print_timings:        eval time =    4764.19 ms /   148 runs   (   32.19 ms per token,    31.07 tokens per second)
llama_print_timings:       to

Groundedness Score:  Steps for evaluation:
1. Identify if all elements of the sepsis management protocol mentioned in the context are included in the answer.
2. Verify that the information in the answer is derived solely from the context provided.

Explanation:
The answer correctly lists all components of sepsis management protocol as per the context, including fluid resuscitation, antibiotics, surgical intervention, supportive care, intensive control of blood glucose, corticosteroids, activated protein C, monitoring of various parameters, and use of vasopressors if necessary. The answer is derived entirely from the information in the context, so the metric is followed completely.

Score: 5
Relevance Score:  The AI-generated answer follows the metric mostly (Score: 4). The answer summarizes the key components of managing sepsis in a critical care unit as stated in the Merck Manual context, including fluid resuscitation, antibiotics, surgical intervention, supportive care, monitoring, a

**Observations for Query 1 (Sepsis) Evaluation:**
- **Groundedness Score: 5** - The evaluation indicates that the AI-generated answer is completely derived from the provided context in the medical manual. All elements of the sepsis management protocol, including specific fluid resuscitation targets (CVP/PAOP), oxygen administration, antibiotics, source control, glucose normalization, and corticosteroids, were found to be present and accurate based on the context.
- **Relevance Score: 4** - The evaluation shows that the answer follows the relevance metric mostly. While the answer correctly summarizes key components of sepsis management, the rater noted that it could have been more concise and directly addressed the specific protocol without unnecessary repetition from the context. This suggests that while the information is relevant, its presentation could be slightly optimized for brevity.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

**Recommended parameters for Query 2 (Appendicitis) in evaluation:**
For a structured answer that attempts to cover symptoms, cure, and surgical procedures, while allowing for some linguistic variation:
- `max_tokens=350` (while it might still truncate slightly, it's a balance)
- `temperature=0.2` (for slight variation while maintaining factual consistency)
- `top_p=0.9`
- `top_k=40`

In [57]:
groundedness_score, relevance_score = generate_ground_relevance_response(
    query_2,
    k=2,
    max_tokens=350,
    temperature=0.2,
    top_p=0.9,
    top_k=40
)
print(f"Groundedness Score: {groundedness_score}")
print(f"Relevance Score: {relevance_score}")

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     205.89 ms /   350 runs   (    0.59 ms per token,  1699.93 tokens per second)
llama_print_timings: prompt eval time =    4481.29 ms /  3425 tokens (    1.31 ms per token,   764.29 tokens per second)
llama_print_timings:        eval time =   11271.68 ms /   349 runs   (   32.30 ms per token,    30.96 tokens per second)
llama_print_timings:       total time =   18129.71 ms /  3774 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      64.68 ms /   124 runs   (    0.52 ms per token,  1917.01 tokens per second)
llama_print_timings: prompt eval time =    5145.43 ms /  3837 tokens (    1.34 ms per token,   745.71 tokens per second)
llama_print_timings:        eval time =    4245.02 ms /   123 runs   (   34.51 ms per token,    28.98 tokens per second)
llama_print_timings:       to

Groundedness Score:  Steps for evaluation:
1. Identify if the AI-generated answer mentions only symptoms and signs of appendicitis derived from the context.
2. Check if the AI-generated answer suggests that appendicitis cannot be cured via medicine but requires surgery for treatment based on the context.

Evaluation:
The AI-generated answer mentions all the common symptoms for appendicitis as stated in the context and correctly identifies that appendicitis cannot be cured via medicine alone but requires surgery for treatment. Therefore, the metric is followed completely.

Score: 5
Relevance Score:  Steps for evaluation:
1. Identify the main aspects of the question: common symptoms for appendicitis and if it can be cured via medicine.
2. Check if the answer addresses these aspects: the answer lists common symptoms for appendicitis but does not mention if it can be cured via medicine or not.

Evaluation:
The answer follows the metric to a good extent as it addresses the main aspects of t

**Observations for Query 2 (Appendicitis) Evaluation:**
- **Groundedness Score: 5** - The evaluation indicates that the AI-generated answer is completely derived from the provided medical context. The symptoms listed for appendicitis and the conclusion that medicine alone cannot cure it, requiring surgery, are directly supported by the information in the manual.
- **Relevance Score: 3** - The evaluation suggests that the answer follows the relevance metric to a good extent. While it effectively addresses the common symptoms of appendicitis, it falls short in providing a comprehensive answer regarding whether it can be cured via medicine and the specific surgical procedures, indicating that it could have more thoroughly covered all aspects of the original question.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

**Recommended parameters for Query 3 (Patchy Hair Loss) in evaluation:**
To provide a detailed list of treatments and causes, leveraging a broader context with a slight creative interpretation:
- `max_tokens=350` (aim for a good portion of detailed information)
- `temperature=0.1` (for largely deterministic output, balancing a slightly wider search space for tokens)
- `top_p=0.85`
- `top_k=30`

In [58]:
groundedness_score, relevance_score = generate_ground_relevance_response(
    query_3,
    k=6,
    max_tokens=350,
    temperature=0.1,
    top_p=0.85,
    top_k=30
)
print(f"Groundedness Score: {groundedness_score}")
print(f"Relevance Score: {relevance_score}")

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      80.40 ms /   141 runs   (    0.57 ms per token,  1753.64 tokens per second)
llama_print_timings: prompt eval time =    4282.51 ms /  3301 tokens (    1.30 ms per token,   770.81 tokens per second)
llama_print_timings:        eval time =    4440.58 ms /   140 runs   (   31.72 ms per token,    31.53 tokens per second)
llama_print_timings:       total time =    9585.24 ms /  3441 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      99.46 ms /   196 runs   (    0.51 ms per token,  1970.58 tokens per second)
llama_print_timings: prompt eval time =    4502.45 ms /  3503 tokens (    1.29 ms per token,   778.02 tokens per second)
llama_print_timings:        eval time =    6436.67 ms /   195 runs   (   33.01 ms per token,    30.30 tokens per second)
llama_print_timings:       to

Groundedness Score:  Steps for evaluation:
1. Identify if the AI-generated answer mentions treatments and causes derived solely from the context.
2. Check if the treatments and causes in the answer match those in the context.

Evaluation:
The AI-generated answer mentions corticosteroids, topical anthralin, minoxidil, systemic corticosteroids, topical immunotherapy (diphencyprone or squaric acid dibutylester), and psoralen plus ultraviolet A (PUVA) as effective treatments for sudden patchy hair loss. These treatments are listed in the context. The cause of this condition, alopecia areata, is identified as an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers, which also matches the information in the context. Therefore, the metric is followed completely.

Score: 5
Relevance Score:  Steps for evaluation:
1. Identify the main aspects of the question: effective treatments for sudden patchy hair loss and possible causes.
2. Check if the an

**Observations for Query 3 (Patchy Hair Loss) Evaluation:**
- **Groundedness Score: 5** - The evaluation confirms that the AI-generated answer for patchy hair loss is entirely derived from the provided medical context. The answer correctly identifies alopecia areata as the likely condition and accurately lists specific treatments (corticosteroids, topical anthralin, minoxidil, immunotherapy, PUVA) and causes (autoimmune disorder, genetic susceptibility) as described in the manual.
- **Relevance Score: 5** - The evaluation indicates that the answer completely addresses the main aspects of the question. It thoroughly covers both the effective treatments and the possible causes of sudden patchy hair loss, directly responding to the user's query based on the contextual information.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

**Recommended parameters for Query 4 (Brain Injury) in evaluation:**
For a comprehensive, multi-part answer covering immediate and long-term care, ensure a generous token budget:
- `max_tokens=500` (essential for a thorough, multi-part answer without truncation)
- `temperature=0.3` (allows for some flexibility in language while maintaining accuracy)
- `top_p=0.9`
- `top_k=40`

In [59]:
groundedness_score, relevance_score = generate_ground_relevance_response(
    query_4,
    k=4,
    max_tokens=500,
    temperature=0.3,
    top_p=0.9,
    top_k=40
)
print(f"Groundedness Score: {groundedness_score}")
print(f"Relevance Score: {relevance_score}")

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      89.11 ms /   140 runs   (    0.64 ms per token,  1571.00 tokens per second)
llama_print_timings: prompt eval time =    4092.10 ms /  3233 tokens (    1.27 ms per token,   790.06 tokens per second)
llama_print_timings:        eval time =    4326.41 ms /   139 runs   (   31.13 ms per token,    32.13 tokens per second)
llama_print_timings:       total time =    9505.62 ms /  3372 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      65.60 ms /   120 runs   (    0.55 ms per token,  1829.18 tokens per second)
llama_print_timings: prompt eval time =    4305.36 ms /  3434 tokens (    1.25 ms per token,   797.61 tokens per second)
llama_print_timings:        eval time =    3760.82 ms /   119 runs   (   31.60 ms per token,    31.64 tokens per second)
llama_print_timings:       to

Groundedness Score:  Step 1: Identify the information in the context related to treatments for traumatic brain injury (TBI).
Step 2: Compare this information to the AI-generated answer to ensure that the answer is derived only from the context and not from external sources.

The answer is derived only from the information in the context as it mentions the initial treatments for TBI which include ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, as well as the possibility of surgery for more severe injuries. Therefore, the score is 5.
Relevance Score:  Step 1: Identify the main aspects of the question related to treatments for brain injuries.
The main aspects of the question are identifying treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function.

Step 2: Evaluate the relevance of the answer to these main aspects based on the context provided.
The answer men

**Observations for Query 4 (Brain Injury) Evaluation:**
- **Groundedness Score: 5** - The evaluation indicates that the AI-generated answer is completely derived from the provided context in the medical manual. The answer accurately identifies initial treatments for traumatic brain injury (TBI), such as ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, and mentions the possibility of surgery for severe injuries. All these details are directly supported by the context.
- **Relevance Score: 4** - The evaluation suggests that the answer follows the relevance metric mostly. While the answer effectively addresses the main aspects of the question by detailing initial treatments for TBI, it doesn't provide a comprehensive list of all specific treatments or long-term considerations that might be inferred from a broader reading of the context. However, the provided information is accurate and directly relevant to the core of the query.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

**Recommended parameters for Query 5 (Fractured Leg) in evaluation:**
While the previous RAG attempt for this query suffered from irrelevant context retrieval (snakebites instead of fractured leg), the parameters for the *fine-tuning combination* were set to explore a broader retrieval and moderate creativity. When evaluating, use the following parameters to reflect the behavior of that specific combination, but note that the underlying retrieval issue still needs addressing.
- `max_tokens=500` (to allow for a more extensive response if relevant context is found)
- `temperature=0.15` (for a balance of determinism and moderate creativity)
- `top_p=0.92`
- `top_k=45`

In [60]:
groundedness_score, relevance_score = generate_ground_relevance_response(
    query_5,
    k=6,
    max_tokens=500,
    temperature=0.15,
    top_p=0.92,
    top_k=45
)
print(f"Groundedness Score: {groundedness_score}")
print(f"Relevance Score: {relevance_score}")

Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =     123.95 ms /   214 runs   (    0.58 ms per token,  1726.43 tokens per second)
llama_print_timings: prompt eval time =    3641.70 ms /  2936 tokens (    1.24 ms per token,   806.22 tokens per second)
llama_print_timings:        eval time =    6541.43 ms /   213 runs   (   30.71 ms per token,    32.56 tokens per second)
llama_print_timings:       total time =   11536.92 ms /  3149 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     399.79 ms
llama_print_timings:      sample time =      39.53 ms /    78 runs   (    0.51 ms per token,  1973.33 tokens per second)
llama_print_timings: prompt eval time =    4132.53 ms /  3211 tokens (    1.29 ms per token,   777.01 tokens per second)
llama_print_timings:        eval time =    2441.84 ms /    77 runs   (   31.71 ms per token,    31.53 tokens per second)
llama_print_timings:       to

Groundedness Score:  Score: 3

The answer follows the metric to a good extent as it derives the information on supportive care and treatment steps for a person who has fractured their leg during hiking from the context. However, it does not explicitly mention the necessary precautions for transporting the person or the importance of rapid transportation to the nearest medical facility as stated in the context.
Relevance Score:  Score: 4

The answer follows the metric mostly by addressing the main aspects of the question related to supportive care for a person who has fractured their leg during hiking. It refers to keeping the person warm, reassured, and transported to a medical facility while immobilizing and elevating the leg for pain management and swelling reduction. The answer also emphasizes the importance of seeking medical attention as soon as possible for proper assessment and treatment of the fracture. However, it does not explicitly mention the necessary precautions for trans

**Observations for Query 5 (Fractured Leg) Evaluation:**
- **Groundedness Score: 3** - The evaluation indicates that the AI-generated answer is followed to a good extent. While the answer derives information on supportive care and initial treatment steps from the context, it fails to explicitly mention crucial precautions like the importance of rapid transportation to a medical facility. This suggests that while some relevant information was retrieved and used, the answer was not fully grounded in all aspects of the context that pertain to the question.
- **Relevance Score: 4** - The evaluation suggests that the answer follows the relevance metric mostly. It effectively addresses general supportive care (keeping warm, reassurance, immobilization) for a fractured leg, emphasizing the need for medical attention. However, it does not provide comprehensive treatment steps beyond basic first aid or specific precautions related to transportation that were present in the context. This indicates that while the answer is mostly relevant, it could be more thorough in addressing all parts of the user's question with the available contextual information.

## Actionable Insights and Business Recommendations

<font size=6 color='blue'>Power Ahead</font>
___

## Actionable Insights and Business Recommendations

### Overall RAG System Effectiveness

Our evaluation demonstrates that the RAG-based AI solution significantly enhances the accuracy, specificity, and completeness of responses to medical queries, especially when compared to the LLM-only baseline. By grounding the LLM's responses in the Merck Manual, the system moves from generic, textbook answers to clinically relevant and detailed protocols.

### Impact of Parameters and Fine-tuning

1.  **Context Length (`max_tokens`)**: A sufficient `max_tokens` (e.g., 250-500) is crucial for comprehensive answers, particularly for multi-part medical questions or detailed protocols. Truncation remains a challenge with smaller token limits.
2.  **Temperature**: Lower temperatures (0-0.2) are ideal for factual, deterministic medical information, ensuring consistency and accuracy, which is paramount in healthcare.
3.  **`top_p` and `top_k`**: These parameters, when tuned slightly higher (e.g., `top_p=0.9-0.95`, `top_k=40-50`), allow for richer, more natural language while still adhering to the factual constraints, contributing to better readability without compromising accuracy.
4.  **Retriever `k`**: Adjusting the number of retrieved documents (`k`) directly impacts the depth and breadth of the answer. Increasing `k` (e.g., to 6 for patchy hair loss) can provide more exhaustive information, but also risks introducing irrelevant context if the embedding model struggles.
5.  **Chunking Strategy**: For the queries tested, changing chunk sizes (1000/100 vs. 500/50) did not drastically alter the response quality for well-covered topics. However, smaller chunks might be beneficial for highly granular information retrieval, and vice-versa for broader topics, warranting further investigation.

### Strengths of the Solution

*   **Medical Specificity**: The RAG system excels at providing highly specific medical details, such as diagnostic criteria, precise treatment protocols, and drug names, directly from the source material.
*   **Structured Output**: Prompt engineering (role-prompting, structured headings, numbered lists) combined with RAG effectively guides the model to produce organized and actionable information, which is invaluable for medical professionals.
*   **Transparency/Groundedness**: The LLM-as-a-judge evaluation confirms high groundedness scores (mostly 5s), indicating that the answers are reliably sourced from the medical manual, fostering trust in the AI's output.

### Weaknesses and Areas for Improvement

*   **Retrieval Accuracy (Critical Failure)**: The most significant weakness is the failure of the retriever for Query 5 (fractured leg), where irrelevant context (snakebites) was provided. This highlights a critical need to enhance the embedding model's robustness or the retriever's logic, especially for nuanced or less directly mapped queries.
*   **Completeness vs. Conciseness**: While higher `max_tokens` improves completeness, achieving optimal balance between a comprehensive and concise answer remains a fine-tuning challenge, as seen with Query 1's relevance score.
*   **Coverage Limitations**: The system is limited by the content of the provided manual. If a topic is not well-covered or if the terminology differs significantly from the query, retrieval accuracy can suffer.

### Business Recommendations

1.  **Iterative Retrieval Enhancement**: Prioritize improving the retrieval component, possibly through:
    *   **Advanced Embedding Models**: Explore fine-tuning embedding models on medical corpora or using more specialized models.
    *   **Hybrid Retrieval**: Implement a hybrid approach combining semantic search with keyword search to improve recall, especially for specific medical terms.
    *   **Reranking**: Integrate a reranking mechanism for retrieved documents to ensure the most relevant chunks are prioritized before generation.

2.  **Adaptive Token Allocation**: Implement dynamic `max_tokens` adjustment based on query complexity or anticipated response length, rather than fixed limits, to avoid truncation for complex queries while maintaining conciseness for simple ones.

3.  **Continuous Content Curation**: Regularly update and expand the medical manual content, ensuring comprehensive coverage of diverse medical topics to minimize instances of irrelevant context retrieval.

4.  **User Feedback Loop for Evaluation**: Establish a system for healthcare professionals to provide direct feedback on AI-generated responses. This can serve as a valuable source for identifying areas where the RAG system needs further training or refinement.

5.  **Integration into Clinical Workflows**: Develop user-friendly interfaces for seamless integration into existing clinical information systems, allowing quick access to AI-powered medical insights at the point of care.

6.  **Scalability and Performance Optimization**: For real-world deployment, optimize the system for speed and scalability, ensuring rapid response times even under heavy load, which is critical in fast-paced healthcare environments.

By addressing these insights and recommendations, healthcare centers can further develop this RAG-based AI solution into a robust tool that significantly aids diagnostic assistance, drug information, treatment planning, and critical care protocols, ultimately improving patient outcomes and operational efficiency.